# Qwen3-VL-8B VQA 로컬 실험 노트북 — RTX 5060 Ti 16GB

목표: public **0.95382 → 0.98**. 이 노트북은 원래 코드가 없어 새로 만든 독립 실험이다. `valid`로 고정 검증을 먼저 수행하고, 개선이 확인된 설정만 `submit`으로 재학습한다. 모든 추론은 로컬 GPU에서 실행하며 외부 추론 API를 호출하지 않는다.

**준비:** VS Code에서 이 파일을 열고 `C:\SSAFY\AIChallenge\baseline\Scripts\python.exe`를 Jupyter 커널로 선택한다. 현재 `dataset` 폴더를 그대로 사용한다. Qwen3-VL-8B 가중치는 `downloads/models/Qwen3-VL-8B-Instruct`에 받았다. 다운로드는 가중치 확보용이며 추론 API가 아니다.

**GPU:** 현재 다른 Python 작업이 약 14.6GB를 사용 중이다. 8B 모델 로딩 전에 12GB 이상 여유가 있어야 하며, 이 노트북은 기존 작업을 종료하지 않는다. 학습 중 OOM이 나면 `MAX_VISUAL_TOKENS=192`로 낮춰 smoke test를 반복한다. 정확도 실험에서는 검증 점수를 보고 해상도를 다시 높인다.

공식 참고: [Qwen3-VL](https://huggingface.co/docs/transformers/main/model_doc/qwen3_vl), [TRL VLM 학습](https://huggingface.co/docs/trl/sft_trainer).

In [1]:
import sys
from importlib.metadata import version
print("QWEN3VL_5060TI_LOCAL_V2: warmup_steps=0.05")
print("Python:", sys.executable)
for package in ("torch", "transformers", "trl", "peft", "bitsandbytes", "datasets", "pandas", "Pillow"):
    print(f"{package}: {version(package)}")

QWEN3VL_5060TI_LOCAL_V2: warmup_steps=0.05
Python: c:\SSAFY\AIChallenge\baseline\Scripts\python.exe
torch: 2.12.1+cu130
transformers: 5.17.0
trl: 0.24.0
peft: 0.18.1
bitsandbytes: 0.50.2
datasets: 4.0.0
pandas: 3.0.6
Pillow: 12.3.0


## 1. 설정

처음에는 `SMOKE=16`으로 데이터와 loss masking을 확인한다. 전체 실험은 `SMOKE=0`으로 다시 실행한다. `MODE="valid"`에서 채택할 설정을 결정한 뒤, **새 커널**에서 `MODE="submit"`으로 전체 train을 재학습한다.

In [2]:
"""Reproducible VQA experiment for a Windows RTX 5060 Ti 16GB.

Uses the existing baseline virtual environment and downloads only model weights.
Run: baseline/Scripts/python.exe qwen3vl_5060ti.py --data dataset --out output/local_valid --mode valid
Then, only after selecting a configuration on validation:
     baseline/Scripts/python.exe qwen3vl_5060ti.py --data dataset --out output/local_final --mode submit
No inference API is used. The Hugging Face Hub only supplies model weights.
"""

import argparse
import csv
import math
import random
from collections import Counter
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset, Image as DatasetImage, Sequence
from peft import LoraConfig, prepare_model_for_kbit_training
from PIL import Image, ImageOps
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from trl import SFTConfig, SFTTrainer


CHOICES = "abcd"
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
SYSTEM = "이미지와 질문을 보고 정답을 고르세요. 이미지 속 작은 글자와 숫자를 주의 깊게 읽으세요. a, b, c, d 중 한 글자만 답하세요."

c:\SSAFY\AIChallenge\baseline\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ROOT = Path(r"C:\SSAFY\AIChallenge")
DATA_DIR = ROOT / "dataset"
MODE = "valid"                    # "valid" or "submit"
SMOKE = 0                        # 0 for a full run
OUT = ROOT / "output" / ("qwen3vl8b_smoke" if SMOKE else f"qwen3vl8b_{MODE}")
LOCAL_MODEL_DIR = ROOT / "downloads" / "models" / "Qwen3-VL-8B-Instruct"
MODEL_ID = str(LOCAL_MODEL_DIR) if (LOCAL_MODEL_DIR / "config.json").exists() else "Qwen/Qwen3-VL-8B-Instruct"
MAX_VISUAL_TOKENS = 256
EPOCHS = 2
LEARNING_RATE = 5e-5
SEED = 42

assert torch.cuda.is_available(), "CUDA GPU가 필요합니다."
assert torch.cuda.is_bf16_supported(), "BF16을 지원하는 GPU가 필요합니다."
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f"GPU 여유 메모리: {free_bytes / 2**30:.1f}/{total_bytes / 2**30:.1f} GiB")
assert free_bytes >= 12 * 2**30, "기존 GPU 작업을 마친 뒤 12 GiB 이상 여유가 있을 때 실행하세요."
for required in ("train.csv", "dev.csv", "test.csv", "sample_submission.csv"):
    assert (DATA_DIR / required).exists(), f"Missing {DATA_DIR / required}"
OUT.mkdir(parents=True, exist_ok=True)
print(torch.cuda.get_device_name(0), "data:", DATA_DIR, "output:", OUT)

GPU 여유 메모리: 14.8/15.9 GiB
NVIDIA GeForce RTX 5060 Ti data: C:\SSAFY\AIChallenge\dataset output: C:\SSAFY\AIChallenge\output\qwen3vl8b_valid


## 2. 분할과 dev 정제

train은 질문 문자열 기준으로 묶고, 100개의 고정 시드 후보 중 정답 분포가 가장 비슷한 약 10%를 검증에 사용한다. validation과 같은 질문, 매우 가까운 이미지는 training fold와 dev 후보에서 제외한다. dev는 3표 이상·낮은 응답 엔트로피만 사용한다.

In [4]:
def normalized_question(value):
    return " ".join(str(value).casefold().split())


def image_hash(path):
    """Small difference hash; used only to keep near-identical images out of validation training."""
    with Image.open(path) as image:
        pixels = list(ImageOps.grayscale(image).resize((9, 8)).getdata())
    result = 0
    for y in range(8):
        for x in range(8):
            result = (result << 1) | (pixels[y * 9 + x] > pixels[y * 9 + x + 1])
    return result


def avoid_validation_leakage(train, valid, data_root):
    valid_questions = set(valid.question.map(normalized_question))
    valid_hashes = [image_hash(data_root / path) for path in valid.path]
    keep = []
    for row in train.itertuples():
        if normalized_question(row.question) in valid_questions:
            keep.append(False)
            continue
        signature = image_hash(data_root / row.path)
        keep.append(all((signature ^ other).bit_count() > 3 for other in valid_hashes))
    return train.loc[keep].reset_index(drop=True)


def dev_consensus(dev):
    selected = []
    for row in dev.itertuples():
        votes = [str(getattr(row, f"answer{i}")).lower() for i in range(1, 6)]
        votes = [vote for vote in votes if vote in ("a", "b", "c", "d")]
        if len(votes) < 3:
            continue
        counts = Counter(votes)
        answer, count = counts.most_common(1)[0]
        entropy = -sum((n / len(votes)) * math.log(n / len(votes)) for n in counts.values())
        entropy /= math.log(4)
        if count >= 3 and count / len(votes) >= 0.6 and entropy <= 0.5:
            selected.append({key: getattr(row, key) for key in ("id", "path", "question", *CHOICES)} | {"answer": answer})
    return pd.DataFrame(selected)


def make_split(train):
    groups = {}
    for index, question in enumerate(train.question.map(normalized_question)):
        groups.setdefault(question, []).append(index)
    overall = train.answer.value_counts(normalize=True).reindex(list(CHOICES), fill_value=0)
    target_size = round(len(train) * 0.1)
    best = None
    for trial in range(100):
        items = list(groups.values())
        random.Random(42 + trial).shuffle(items)
        held_out = []
        for indices in items:
            if len(held_out) >= target_size:
                break
            held_out.extend(indices)
        proportions = train.iloc[held_out].answer.value_counts(normalize=True).reindex(list(CHOICES), fill_value=0)
        distance = (proportions - overall).abs().sum() + abs(len(held_out) - target_size) / len(train)
        if best is None or distance < best[0]:
            best = (distance, set(held_out))
    fit = train.iloc[[index for index in range(len(train)) if index not in best[1]]]
    valid = train.iloc[sorted(best[1])]
    return fit.reset_index(drop=True), valid.reset_index(drop=True)

In [5]:
random.seed(SEED)
torch.manual_seed(SEED)
train_df = pd.read_csv(DATA_DIR / "train.csv").fillna("")
test_df = pd.read_csv(DATA_DIR / "test.csv").fillna("")
if MODE == "valid":
    fit_df, valid_df = make_split(train_df)
    valid_df[["id", "path", "question", "answer"]].to_csv(OUT / "valid_split.csv", index=False)
    fit_df = avoid_validation_leakage(fit_df, valid_df, DATA_DIR)
else:
    fit_df, valid_df = train_df, None

dev_df = dev_consensus(pd.read_csv(DATA_DIR / "dev.csv").fillna(""))
if valid_df is not None:
    dev_df = avoid_validation_leakage(dev_df, valid_df, DATA_DIR)
fit_df = pd.concat([fit_df, dev_df], ignore_index=True)
if SMOKE:
    fit_df = fit_df.sample(n=min(SMOKE, len(fit_df)), random_state=SEED).reset_index(drop=True)
    if valid_df is not None:
        valid_df = valid_df.head(10)
print("train rows:", len(fit_df), "dev added:", len(dev_df), "validation rows:", 0 if valid_df is None else len(valid_df))
assert fit_df.answer.isin(list(CHOICES)).all()

C:\Users\SSAFY\AppData\Local\Temp\ipykernel_19648\308800820.py:8: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = list(ImageOps.grayscale(image).resize((9, 8)).getdata())
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_19648\308800820.py:8: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = list(ImageOps.grayscale(image).resize((9, 8)).getdata())


train rows: 7305 dev added: 1263 validation rows: 671


## 3. 모델과 학습

Qwen3-VL-8B 4bit LoRA를 사용한다. language model 모듈만 대상으로 LoRA를 적용한다. 손실은 assistant의 정답 글자 한 토큰에만 계산하며, 첫 batch에서 토큰 수를 검사한다. 16GB VRAM에 맞춰 batch size 1과 gradient checkpointing을 사용한다.

In [6]:
def prompt(row):
    return "질문: {}\n(a) {}\n(b) {}\n(c) {}\n(d) {}\n정답:".format(
        row["question"], row["a"], row["b"], row["c"], row["d"]
    )


def messages(row, image=None, answer=None):
    image_part = {"type": "image"}
    if image is not None:
        image_part["image"] = image
    result = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
        {"role": "user", "content": [image_part, {"type": "text", "text": prompt(row)}]},
    ]
    if answer is not None:
        result.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return result


def training_dataset(frame, data_root):
    records = []
    for row in frame.to_dict("records"):
        records.append({
            "images": [str((data_root / row["path"]).resolve())],
            "messages": messages(row, answer=row["answer"]),
        })
    return Dataset.from_list(records).cast_column("images", Sequence(DatasetImage()))


def make_answer_only_collator(base_collator, token_ids):
    allowed = torch.tensor(token_ids)

    def collate(features):
        batch = base_collator(features)
        labels = batch["labels"]
        original = labels.clone()
        labels.fill_(-100)
        for row, feature in enumerate(features):
            end = int(batch["attention_mask"][row].sum())
            positions = torch.where(torch.isin(original[row, :end], allowed))[0]
            if len(positions) == 0 or end - int(positions[-1]) > 4:
                raise RuntimeError("Could not locate the assistant answer token near the sequence end.")
            answer = feature["messages"][-1]["content"][0]["text"]
            expected_id = token_ids[CHOICES.index(answer)]
            if int(original[row, positions[-1]]) != expected_id:
                raise RuntimeError("Last choice token does not match the training answer.")
            labels[row, positions[-1]] = expected_id
        if (labels != -100).sum().item() != len(features):
            raise RuntimeError("Expected exactly one supervised answer token per example.")
        return batch

    return collate


def load_model(max_tokens):
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    processor.image_processor.size["shortest_edge"] = 64 * 32 * 32
    processor.image_processor.size["longest_edge"] = max_tokens * 32 * 32
    processor.tokenizer.padding_side = "right"
    ids = [processor.tokenizer.encode(c, add_special_tokens=False) for c in CHOICES]
    if any(len(item) != 1 for item in ids):
        raise ValueError(f"Choice letters must be single tokens: {ids}")
    token_ids = [item[0] for item in ids]
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        ),
        device_map={"": 0},
        dtype=torch.bfloat16,
        attn_implementation="sdpa",
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    target_names = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
    targets = [name for name, module in model.named_modules()
               if "language_model" in name and name.rsplit(".", 1)[-1] in target_names
               and isinstance(module, torch.nn.Linear)]
    if not targets:
        raise RuntimeError("No language-model LoRA modules found; inspect model.named_modules().")
    print(f"LoRA target modules: {len(targets)}; examples: {targets[:3]}")
    return model, processor, token_ids, targets

In [ ]:
model, processor, token_ids, targets = load_model(MAX_VISUAL_TOKENS)
config = SFTConfig(
    output_dir=str(OUT / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=LEARNING_RATE,
    warmup_steps=0.05,
    lr_scheduler_type="linear",
    bf16=True,
    max_length=None,
    assistant_only_loss=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False,
    save_strategy="epoch",
    logging_steps=20,
    report_to="none",
    seed=SEED,
)
trainer = SFTTrainer(
    model=model, args=config,
    train_dataset=training_dataset(fit_df, DATA_DIR),
    processing_class=processor,
    peft_config=LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                           target_modules=targets, task_type="CAUSAL_LM"),
)
trainer.data_collator = make_answer_only_collator(trainer.data_collator, token_ids)
probe = trainer.data_collator([trainer.train_dataset[0]])
assert (probe["labels"] != -100).sum().item() == 1
print("Answer-token loss mask verified.")

W0922 09:20:20.291000 19648 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 750/750 [00:12<00:00, 59.79it/s] 


LoRA target modules: 252; examples: ['model.language_model.layers.0.self_attn.q_proj', 'model.language_model.layers.0.self_attn.k_proj', 'model.language_model.layers.0.self_attn.v_proj']


Casting the dataset: 100%|██████████| 7305/7305 [00:00<00:00, 608818.32 examples/s]


Answer-token loss mask verified.


: 

In [ ]:
trainer.train()
trainer.model.save_pretrained(OUT / "adapter")
processor.save_pretrained(OUT / "adapter")
print("Adapter saved:", OUT / "adapter")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.
c:\SSAFY\AIChallenge\baseline\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


Step,Training Loss
20,1.111720
40,1.034360
60,0.544919
80,0.484629
100,0.444766
120,0.408647
140,0.448047
160,0.515527
180,0.432954
200,0.417310


c:\SSAFY\AIChallenge\baseline\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


## 4. 검증 점수 또는 제출 파일

문항마다 a/b/c/d log probability를 저장한다. 검증 점수를 먼저 확인한다. `SMOKE` 실행에서는 제출 파일을 만들지 않는다.

In [ ]:
@torch.inference_mode()
def score(frame, model, processor, token_ids, data_root, output):
    model.eval()
    model.config.use_cache = True
    device = next(model.parameters()).device
    answer_ids = torch.tensor(token_ids, device=device)
    columns = ["id", "path", "question", "gold", "pred", "logp_a", "logp_b", "logp_c", "logp_d"]
    correct = 0
    with output.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        for index, row in enumerate(frame.to_dict("records"), 1):
            with Image.open(data_root / row["path"]) as image:
                chat = messages(row, image=image.convert("RGB"))
                inputs = processor.apply_chat_template(
                    chat, tokenize=True, add_generation_prompt=True,
                    return_dict=True, return_tensors="pt",
                )
            inputs = {key: value.to(device) if torch.is_tensor(value) else value
                      for key, value in inputs.items()}
            with torch.autocast("cuda", dtype=torch.bfloat16):
                logits = model(**inputs).logits[0, -1].index_select(0, answer_ids)
            logp = torch.log_softmax(logits.float(), dim=0).cpu().tolist()
            predicted = CHOICES[max(range(4), key=logp.__getitem__)]
            gold = row.get("answer", "")
            correct += predicted == gold
            writer.writerow(dict(zip(columns, [row["id"], row["path"], row["question"],
                                               gold, predicted, *logp])))
            if index % 100 == 0:
                print(f"Scored {index}/{len(frame)}", flush=True)
    if "answer" in frame:
        print(f"Validation: {correct}/{len(frame)} = {correct / len(frame):.5f}")

In [ ]:
if MODE == "valid":
    score(valid_df, trainer.model, processor, token_ids, DATA_DIR, OUT / "valid_scores.csv")
elif not SMOKE:
    score(test_df, trainer.model, processor, token_ids, DATA_DIR, OUT / "test_scores.csv")
    scores = pd.read_csv(OUT / "test_scores.csv")
    sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
    assert len(sample) == len(test_df) and sample.id.equals(test_df.id)
    sample["answer"] = scores.pred
    assert sample.answer.isin(list(CHOICES)).all()
    sample.to_csv(OUT / "submission.csv", index=False)
    print("Submission saved:", OUT / "submission.csv")

c:\SSAFY\AIChallenge\baseline\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


Validation: 8/10 = 0.80000


## 5. 검증 오답과 다음 실험

검증에서는 실제 오답을 CSV로 남긴다. `RUN_HIGHRES_ABLATION=True`로 바꾸면 가장 불확실한 15%만 높은 해상도로 다시 읽고, **같은 검증 행에서 정답 수가 늘었는지** 확인한다. 개선이 없으면 제출에 적용하지 않는다.

In [ ]:
if MODE == "valid" and not SMOKE:
    scored = pd.read_csv(OUT / "valid_scores.csv")
    errors = scored.loc[scored.gold != scored.pred].copy()
    errors.to_csv(OUT / "valid_errors.csv", index=False)
    val_accuracy = (scored.gold == scored.pred).mean()
    pd.DataFrame([{
        "model": MODEL_ID, "seed": SEED, "train_rows": len(fit_df),
        "dev_added": len(dev_df), "valid_rows": len(scored),
        "max_visual_tokens": MAX_VISUAL_TOKENS, "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE, "val_accuracy": val_accuracy,
    }]).to_csv(OUT / "experiment_summary.csv", index=False)
    print(f"Validation: {(scored.gold == scored.pred).sum()}/{len(scored)} = {val_accuracy:.5f}")
    print("Errors:", len(errors), "saved:", OUT / "valid_errors.csv")
    display(errors[["id", "question", "gold", "pred"]].head(30))

In [ ]:
RUN_HIGHRES_ABLATION = False
if MODE == "valid" and not SMOKE and RUN_HIGHRES_ABLATION:
    scored = pd.read_csv(OUT / "valid_scores.csv")
    logp = scored[[f"logp_{c}" for c in CHOICES]].to_numpy()
    margins = pd.Series([sorted(row)[-1] - sorted(row)[-2] for row in logp], index=scored.id)
    selected_ids = set(margins.nsmallest(max(1, round(len(scored) * 0.15))).index)
    selected = valid_df.loc[valid_df.id.isin(selected_ids)].copy()
    old_budget = processor.image_processor.size["longest_edge"]
    processor.image_processor.size["longest_edge"] = 512 * 32 * 32
    try:
        score(selected, trainer.model, processor, token_ids, DATA_DIR, OUT / "valid_highres_subset.csv")
    finally:
        processor.image_processor.size["longest_edge"] = old_budget
    replacement = pd.read_csv(OUT / "valid_highres_subset.csv").set_index("id")
    combined = scored.set_index("id")
    combined.update(replacement)
    combined.reset_index().to_csv(OUT / "valid_scores_highres_selective.csv", index=False)
    old_correct = (scored.gold == scored.pred).sum()
    new_correct = (combined.gold == combined.pred).sum()
    print(f"Base {old_correct}/{len(scored)} → selective high-res {new_correct}/{len(scored)}; delta {new_correct-old_correct}")